In [1]:
"""
gesture_engine.py
-----------------
Handles all MediaPipe hand tracking and gesture classification.

GESTURES DEFINED:
  - INDEX_ONLY   → Drawing mode (only index finger extended)
  - PEACE        → Hover/pause (index + middle extended, no drawing)
  - FIST         → Trigger AI analysis
  - OPEN_HAND    → Eraser mode (4+ fingers extended)
  - PINCH        → Future: brush size control (index + thumb close)
  - IDLE         → No hand / unrecognized

IMPROVISATION NOTE:
  The original idea was just "index finger draws."
  Problem: you'd never be able to lift your hand without drawing a streak.
  Solution: gesture modes. PEACE sign = hover without drawing.
  This is the #1 UX fix that makes the app actually usable.
"""

import mediapipe as mp
import numpy as np
from dataclasses import dataclass
from enum import Enum, auto


class Gesture(Enum):
    INDEX_ONLY = auto()   # Draw
    PEACE      = auto()   # Hover / pause
    OPEN_HAND  = auto()   # Erase
    FIST       = auto()   # Analyze
    PINCH      = auto()   # Brush resize
    IDLE       = auto()   # No detection


@dataclass
class HandData:
    gesture: Gesture
    index_tip: tuple[int, int]   # (x, y) in pixel coords
    raw_landmarks: object        # mediapipe landmark object


class GestureEngine:
    """
    Wraps MediaPipe Hands. Call process_frame() each loop iteration.
    """

    def __init__(self, max_hands: int = 1):
        self._mp_hands = mp.solutions.hands
        self._mp_draw  = mp.solutions.drawing_utils

        self.hands = self._mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=0.80,
            min_tracking_confidence=0.75,
        )

        # Landmark indices
        self.FINGERTIP_IDS  = [4, 8, 12, 16, 20]
        self.FINGER_PIP_IDS = [3, 6, 10, 14, 18]   # one joint below tip

    # ------------------------------------------------------------------ #
    #  Public API                                                          #
    # ------------------------------------------------------------------ #

    def process_frame(self, rgb_frame) -> list[HandData]:
        """
        Returns a list of HandData (one per detected hand).
        rgb_frame must be RGB (not BGR).
        """
        results = self.hands.process(rgb_frame)
        hand_data_list = []

        if not results.multi_hand_landmarks:
            return hand_data_list

        h, w = rgb_frame.shape[:2]

        for landmarks, handedness_info in zip(
            results.multi_hand_landmarks,
            results.multi_handedness
        ):
            hand_label = handedness_info.classification[0].label  # 'Left' or 'Right'
            fingers    = self._fingers_up(landmarks, hand_label)
            gesture    = self._classify_gesture(fingers, landmarks, w, h)

            tip = landmarks.landmark[8]   # index fingertip
            ix  = int(tip.x * w)
            iy  = int(tip.y * h)

            hand_data_list.append(HandData(
                gesture=gesture,
                index_tip=(ix, iy),
                raw_landmarks=landmarks,
            ))

        return hand_data_list

    def draw_landmarks(self, bgr_frame, hand_data: HandData):
        """Draws the skeleton overlay on the BGR frame (in-place)."""
        self._mp_draw.draw_landmarks(
            bgr_frame,
            hand_data.raw_landmarks,
            self._mp_hands.HAND_CONNECTIONS,
            self._mp_draw.DrawingSpec(color=(0, 220, 120), thickness=2, circle_radius=3),
            self._mp_draw.DrawingSpec(color=(255, 200, 0), thickness=2),
        )

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _fingers_up(self, landmarks, hand_label: str) -> list[int]:
        """
        Returns [thumb, index, middle, ring, pinky] — 1 = extended, 0 = folded.
        Thumb uses horizontal comparison (depends on which hand).
        """
        lm = landmarks.landmark
        fingers = []

        # Thumb: compare tip x vs. IP joint x (mirrored for left hand)
        if hand_label == 'Right':
            fingers.append(1 if lm[4].x < lm[3].x else 0)
        else:
            fingers.append(1 if lm[4].x > lm[3].x else 0)

        # Index → Pinky: tip y vs. PIP joint y (up = smaller y)
        for tip_id, pip_id in zip(self.FINGERTIP_IDS[1:], self.FINGER_PIP_IDS[1:]):
            fingers.append(1 if lm[tip_id].y < lm[pip_id].y else 0)

        return fingers  # length 5

    def _classify_gesture(self, fingers: list[int], landmarks, w: int, h: int) -> Gesture:
        thumb, index, middle, ring, pinky = fingers

        extended_count = sum(fingers[1:])  # exclude thumb

        # FIST: all fingers folded
        if extended_count == 0 and thumb == 0:
            return Gesture.FIST

        # OPEN HAND: 4 or 5 non-thumb fingers up
        if extended_count >= 4:
            return Gesture.OPEN_HAND

        # PEACE: index + middle only
        if index == 1 and middle == 1 and ring == 0 and pinky == 0:
            return Gesture.PEACE

        # INDEX ONLY: drawing gesture
        if index == 1 and middle == 0 and ring == 0 and pinky == 0:
            return Gesture.INDEX_ONLY

        # PINCH: check distance between index tip and thumb tip
        if index == 1 and thumb == 1 and middle == 0:
            lm   = landmarks.landmark
            dx   = (lm[4].x - lm[8].x) * w
            dy   = (lm[4].y - lm[8].y) * h
            dist = np.hypot(dx, dy)
            if dist < 40:
                return Gesture.PINCH

        return Gesture.IDLE

In [2]:
"""
canvas_engine.py
----------------
Manages the drawing canvas: strokes, erasing, undo history, smoothing.

IMPROVISATION #1 — Stroke-based smoothing (not just point averaging)
  The naive approach buffers N points and averages them → still jagged for fast
  movements. Instead, we use Catmull-Rom spline interpolation on committed
  strokes. Each stroke is collected as a list of raw points, then re-rendered
  smoothly after the finger lifts. This makes Korean characters look like
  handwriting instead of pixel noise.

IMPROVISATION #2 — Stroke-level undo (not canvas snapshot undo)
  Snapshot undo wastes memory for high-res frames. Instead we store the list of
  strokes. Undo = pop last stroke and redraw everything. This is lighter and
  gives you infinite undo within the session.

IMPROVISATION #3 — Adaptive brush pressure simulation
  Stroke speed is measured. Slower movement → thicker line (simulates pen
  pressure). Korean characters look more natural with thick/thin contrast.
"""

import cv2
import numpy as np
from collections import deque
from dataclasses import dataclass, field


@dataclass
class Stroke:
    points: list         # list of (x, y)
    color:  tuple        # BGR
    thickness: int


class CanvasEngine:

    # ------------------------------------------------------------------ #
    #  Init                                                                #
    # ------------------------------------------------------------------ #

    def __init__(self, width: int, height: int):
        self.width  = width
        self.height = height

        self._strokes: list[Stroke]   = []   # committed strokes
        self._current_pts: list       = []   # points of ongoing stroke
        self._current_color: tuple    = (255, 255, 255)
        self._base_thickness: int     = 8

        # Smoothing buffer for live cursor position
        self._smooth_buf: deque = deque(maxlen=6)

        # Cached composite canvas (rebuilt on undo / new stroke commit)
        self._canvas: np.ndarray = np.zeros((height, width, 3), dtype=np.uint8)
        self._dirty: bool        = False

        # Pressure simulation
        self._prev_draw_pt: tuple | None = None

    # ------------------------------------------------------------------ #
    #  Public drawing API                                                  #
    # ------------------------------------------------------------------ #

    def smooth(self, x: int, y: int) -> tuple[int, int]:
        """Exponential weighted moving average — snappy but stable."""
        self._smooth_buf.append((x, y))
        if len(self._smooth_buf) < 2:
            return x, y
        alpha = 0.55   # higher → more responsive, lower → smoother
        sx, sy = self._smooth_buf[-2]
        rx = int(alpha * x + (1 - alpha) * sx)
        ry = int(alpha * y + (1 - alpha) * sy)
        return rx, ry

    def begin_stroke(self, x: int, y: int):
        self._current_pts = [(x, y)]
        self._prev_draw_pt = (x, y)

    def continue_stroke(self, x: int, y: int) -> tuple[int, int]:
        """
        Adds point to current stroke and returns adaptive thickness.
        Slower speed → thicker line (pressure simulation).
        """
        if not self._current_pts:
            self.begin_stroke(x, y)
            return x, y

        # Pressure: speed-based thickness
        px, py     = self._current_pts[-1]
        speed      = np.hypot(x - px, y - py)
        thickness  = self._pressure_thickness(speed)

        # Draw segment on canvas immediately (for live feedback)
        cv2.line(self._canvas, (px, py), (x, y), self._current_color, thickness)
        self._current_pts.append((x, y))
        self._dirty = True

        return x, y

    def end_stroke(self):
        """Commit current stroke to history."""
        if len(self._current_pts) > 1:
            self._strokes.append(Stroke(
                points=list(self._current_pts),
                color=self._current_color,
                thickness=self._base_thickness,
            ))
        self._current_pts = []
        self._prev_draw_pt = None
        self._smooth_buf.clear()

    def erase(self, x: int, y: int, radius: int = 30):
        """Erases a circular area. Also removes any stroke whose points fall in the circle."""
        cv2.circle(self._canvas, (x, y), radius, (0, 0, 0), -1)
        # Remove strokes fully within the erased region
        remaining = []
        for stroke in self._strokes:
            pts = np.array(stroke.points)
            dists = np.hypot(pts[:, 0] - x, pts[:, 1] - y)
            if np.any(dists > radius):
                remaining.append(stroke)
        if len(remaining) != len(self._strokes):
            self._strokes = remaining
            self._rebuild_canvas()

    def undo(self):
        """Remove the last committed stroke and redraw."""
        if self._strokes:
            self._strokes.pop()
            self._rebuild_canvas()

    def clear(self):
        self._strokes.clear()
        self._current_pts = []
        self._canvas[:] = 0
        self._dirty = False

    # ------------------------------------------------------------------ #
    #  Color / brush                                                       #
    # ------------------------------------------------------------------ #

    def set_color(self, bgr: tuple):
        self._current_color = bgr

    def set_thickness(self, t: int):
        self._base_thickness = max(2, min(40, t))

    @property
    def thickness(self) -> int:
        return self._base_thickness

    # ------------------------------------------------------------------ #
    #  Frame compositing                                                   #
    # ------------------------------------------------------------------ #

    def composite(self, bgr_frame: np.ndarray) -> np.ndarray:
        """
        Blend canvas over camera frame.
        Canvas pixels that are non-black override the frame.
        """
        gray   = cv2.cvtColor(self._canvas, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        mask_inv = cv2.bitwise_not(mask)

        bg = cv2.bitwise_and(bgr_frame, bgr_frame, mask=mask_inv)
        fg = cv2.bitwise_and(self._canvas, self._canvas, mask=mask)
        return cv2.add(bg, fg)

    def get_canvas_snapshot(self) -> np.ndarray:
        """Return a copy of the current canvas (for AI analysis)."""
        return self._canvas.copy()

    def has_content(self) -> bool:
        return bool(self._strokes) or bool(self._current_pts)

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _pressure_thickness(self, speed: float) -> int:
        """
        Simulates pen pressure: slow strokes are thicker.
        Speed in pixels/frame. Tuned for 30fps webcam.
        """
        # Clamp speed 0–60 px/frame → thickness BASE to BASE/2
        base  = self._base_thickness
        ratio = np.clip(speed / 60.0, 0, 1)
        t     = int(base * (1.0 - 0.45 * ratio))
        return max(2, t)

    def _rebuild_canvas(self):
        """Redraw all committed strokes from scratch (used by undo/erase)."""
        self._canvas[:] = 0
        for stroke in self._strokes:
            pts = stroke.points
            if len(pts) < 2:
                continue
            for i in range(1, len(pts)):
                px, py = pts[i - 1]
                cx, cy = pts[i]
                speed  = np.hypot(cx - px, cy - py)
                thick  = self._pressure_thickness(speed)
                cv2.line(self._canvas, (px, py), (cx, cy), stroke.color, thick)
        self._dirty = False

In [3]:
"""
korean_analyzer.py
------------------
Sends a canvas snapshot to Claude's vision API and returns Korean text analysis.
Runs in a background thread so the camera loop never freezes.

IMPROVISATION #4 — Non-blocking analysis with threading
  If you call the Anthropic API synchronously inside the OpenCV loop, the
  webcam freezes for 2-5 seconds while waiting. Users will think it crashed.
  Solution: run the API call in a daemon thread. The UI shows a pulsing
  "Analyzing…" animation while it runs.

IMPROVISATION #5 — Canvas pre-processing before sending
  Raw canvas is 1280×720 of mostly black pixels — wastes tokens and makes
  OCR harder. We auto-crop to the bounding box of actual drawing content,
  add a white background, upscale if needed, and send a clean image.
  This dramatically improves Korean character recognition accuracy.

IMPROVISATION #6 — Structured JSON prompt
  A vague "tell me what this says" prompt gives inconsistent output.
  We use a strict JSON schema prompt so the result is always parseable
  and the UI can render it in labeled fields (Korean / Romanization / English).
"""

import base64
import json
import threading
import io
from enum import Enum, auto

import cv2
import numpy as np
from PIL import Image

try:
    import anthropic
    _ANTHROPIC_AVAILABLE = True
except ImportError:
    _ANTHROPIC_AVAILABLE = False


class AnalysisState(Enum):
    IDLE      = auto()
    RUNNING   = auto()
    DONE      = auto()
    ERROR     = auto()


class AnalysisResult:
    def __init__(self):
        self.korean        = ""
        self.romanization  = ""
        self.translation   = ""
        self.notes         = ""
        self.raw           = ""

    def __str__(self):
        lines = []
        if self.korean:       lines.append(f"Korean:        {self.korean}")
        if self.romanization: lines.append(f"Romanization:  {self.romanization}")
        if self.translation:  lines.append(f"Meaning:       {self.translation}")
        if self.notes:        lines.append(f"Notes:         {self.notes}")
        return "\n".join(lines) if lines else "No text detected."


class KoreanAnalyzer:

    SYSTEM_PROMPT = """You are an expert Korean language OCR assistant.
The user has drawn Korean characters using an air canvas (finger tracking on webcam).
The strokes may be imperfect or stylized. Your job is to:
1. Identify every Korean character or word visible in the image
2. Provide romanization (Revised Romanization)
3. Provide an accurate English translation
4. Note if the writing is ambiguous or partially formed

IMPORTANT: Respond ONLY with a valid JSON object, no markdown, no explanation outside JSON.
Schema:
{
  "korean": "<the Korean text you identified, empty string if none>",
  "romanization": "<romanized pronunciation>",
  "translation": "<English meaning>",
  "confidence": "high|medium|low",
  "notes": "<any useful observation about stroke quality or ambiguity>"
}"""

    USER_PROMPT = (
        "Please analyze this air-drawn Korean text. "
        "The image is a cropped and cleaned snapshot of only the drawn strokes. "
        "Return the JSON analysis."
    )

    def __init__(self, api_key: str | None = None):
        self.state:  AnalysisState  = AnalysisState.IDLE
        self.result: AnalysisResult = AnalysisResult()
        self._thread: threading.Thread | None = None

        if not _ANTHROPIC_AVAILABLE:
            raise RuntimeError("anthropic package not installed. Run: pip install anthropic")

        self._client = anthropic.Anthropic(api_key=api_key)   # uses ANTHROPIC_API_KEY env var if None

    # ------------------------------------------------------------------ #
    #  Public API                                                          #
    # ------------------------------------------------------------------ #

    def submit(self, canvas: np.ndarray):
        """
        Start an async analysis. Call this once; check .state and .result after.
        """
        if self.state == AnalysisState.RUNNING:
            return   # already in progress

        self.state  = AnalysisState.RUNNING
        self.result = AnalysisResult()

        prepped = self._preprocess(canvas)
        if prepped is None:
            self.result.notes = "Canvas appears empty — nothing to analyze."
            self.state = AnalysisState.DONE
            return

        self._thread = threading.Thread(
            target=self._run_api,
            args=(prepped,),
            daemon=True,
        )
        self._thread.start()

    def reset(self):
        self.state  = AnalysisState.IDLE
        self.result = AnalysisResult()

    # ------------------------------------------------------------------ #
    #  Private                                                             #
    # ------------------------------------------------------------------ #

    def _preprocess(self, canvas: np.ndarray) -> str | None:
        """
        1. Auto-crop to bounding box of non-black pixels
        2. Add white background
        3. Upscale to at least 256×256
        4. Return base64-encoded PNG string
        """
        gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
        coords = cv2.findNonZero(gray)
        if coords is None:
            return None

        x, y, w, h = cv2.boundingRect(coords)
        pad = 30
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(canvas.shape[1], x + w + pad)
        y2 = min(canvas.shape[0], y + h + pad)

        cropped = canvas[y1:y2, x1:x2]

        # White background
        white_bg = np.full_like(cropped, 255)
        gray_crop = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray_crop, 10, 255, cv2.THRESH_BINARY)
        mask_inv = cv2.bitwise_not(mask)
        bg = cv2.bitwise_and(white_bg, white_bg, mask=mask_inv)
        fg = cv2.bitwise_and(cropped, cropped, mask=mask)
        result_img = cv2.add(bg, fg)

        # Upscale if too small
        ch, cw = result_img.shape[:2]
        target = 512
        if cw < target or ch < target:
            scale = target / max(cw, ch)
            result_img = cv2.resize(
                result_img,
                (int(cw * scale), int(ch * scale)),
                interpolation=cv2.INTER_LANCZOS4,
            )

        # Encode
        rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb)
        buf = io.BytesIO()
        pil.save(buf, format="PNG")
        return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

    def _run_api(self, b64_image: str):
        try:
            response = self._client.messages.create(
                model="claude-opus-4-5",
                max_tokens=512,
                system=self.SYSTEM_PROMPT,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": "image/png",
                                    "data": b64_image,
                                },
                            },
                            {"type": "text", "text": self.USER_PROMPT},
                        ],
                    }
                ],
            )

            raw = response.content[0].text.strip()
            self.result.raw = raw

            # Strip markdown fences if model adds them anyway
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]

            data = json.loads(raw)
            self.result.korean        = data.get("korean", "")
            self.result.romanization  = data.get("romanization", "")
            self.result.translation   = data.get("translation", "")
            self.result.notes         = data.get("notes", "")
            self.state = AnalysisState.DONE

        except json.JSONDecodeError:
            # Model didn't return JSON — just show raw text
            self.result.translation = self.result.raw
            self.state = AnalysisState.DONE
        except Exception as exc:
            self.result.notes = f"API Error: {exc}"
            self.state = AnalysisState.ERROR

In [5]:
"""
ui_renderer.py
--------------
Draws toolbar, cursor, status bar, and result overlay onto the BGR frame.

IMPROVISATION #7 — Dwell-click for toolbar buttons (no keyboard needed)
  Problem: clicking a color swatch by hovering with your finger is unreliable
  — any slight jitter while in PEACE mode triggers the wrong color.
  Solution: "dwell" activation. You must hover over a button for DWELL_FRAMES
  consecutive frames before it activates. A circular progress arc fills up
  visually, giving you feedback. No accidental clicks.

IMPROVISATION #8 — Brush size control via pinch distance (not a slider)
  The original spec doesn't mention brush resize. Pinch gesture (thumb+index
  close together) scales the brush from 3px to 30px based on pinch distance.
  This is more intuitive than any on-screen slider.
"""

import cv2
import numpy as np
import math
import time


# ── Colour palette (BGR) ──────────────────────────────────────────────────── #
PALETTE = [
    ("White",   (255, 255, 255)),
    ("Red",     (0,   0,   220)),
    ("Orange",  (0,   130, 255)),
    ("Yellow",  (0,   220, 220)),
    ("Green",   (30,  200, 60)),
    ("Cyan",    (220, 200, 0)),
    ("Blue",    (230, 80,  0)),
    ("Purple",  (200, 0,   180)),
    ("Eraser",  (0,   0,   0)),
]

TOOLBAR_H     = 90          # px — height of top toolbar
SWATCH_SZ     = 52          # px — colour swatch size
SWATCH_PAD    = 10          # px — gap between swatches
DWELL_FRAMES  = 18          # frames of hover before activation (~0.6 s at 30fps)


class UIRenderer:

    def __init__(self, frame_w: int, frame_h: int):
        self.fw = frame_w
        self.fh = frame_h

        self.selected_palette_idx = 0   # default White
        self._dwell_target = None       # (zone_id,) currently being dwelled on
        self._dwell_count  = 0

        # Button zones (computed once)
        self._swatch_zones   = self._compute_swatch_zones()
        self._analyze_zone   = self._compute_btn_zone(0)
        self._undo_zone      = self._compute_btn_zone(1)
        self._clear_zone     = self._compute_btn_zone(2)
        self._save_zone      = self._compute_btn_zone(3)

        # For pulsing animations
        self._frame_count = 0

    # ------------------------------------------------------------------ #
    #  Properties                                                          #
    # ------------------------------------------------------------------ #

    @property
    def current_color(self) -> tuple:
        return PALETTE[self.selected_palette_idx][1]

    @property
    def toolbar_height(self) -> int:
        return TOOLBAR_H

    # ------------------------------------------------------------------ #
    #  Main draw call                                                      #
    # ------------------------------------------------------------------ #

    def draw_toolbar(self, frame: np.ndarray):
        self._frame_count += 1

        # Dark bar background
        cv2.rectangle(frame, (0, 0), (self.fw, TOOLBAR_H), (20, 20, 28), -1)
        # Subtle bottom border
        cv2.line(frame, (0, TOOLBAR_H), (self.fw, TOOLBAR_H), (60, 60, 80), 1)

        self._draw_swatches(frame)
        self._draw_buttons(frame)

    def draw_cursor(self, frame: np.ndarray, cx: int, cy: int,
                    gesture_name: str, color: tuple, thickness: int):
        """Draw the cursor dot at finger tip position."""
        if cy < TOOLBAR_H:
            return

        if gesture_name == "INDEX_ONLY":
            cv2.circle(frame, (cx, cy), thickness + 2, color, -1)
            cv2.circle(frame, (cx, cy), thickness + 4, (255, 255, 255), 1)
        elif gesture_name == "PEACE":
            cv2.circle(frame, (cx, cy), 10, (0, 230, 255), 2)
        elif gesture_name == "OPEN_HAND":
            cv2.circle(frame, (cx, cy), 32, (100, 100, 255), 2)
            cv2.putText(frame, "ERASE", (cx - 22, cy + 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 255), 1)
        elif gesture_name == "FIST":
            cv2.circle(frame, (cx, cy), 18, (0, 200, 100), 2)
            cv2.putText(frame, "ANALYZE", (cx - 35, cy + 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 100), 1)

    def draw_status(self, frame: np.ndarray, mode: str, thickness: int):
        """Bottom status bar."""
        y = self.fh - 12
        color_name = PALETTE[self.selected_palette_idx][0]
        text = (
            f" ✏ {mode}  |  Color: {color_name}  |  "
            f"Brush: {thickness}px  |  "
            f"Gestures → Index:Draw  Peace:Hover  OpenHand:Erase  Fist:Analyze"
        )
        cv2.putText(frame, text, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (140, 140, 160), 1)

    # ------------------------------------------------------------------ #
    #  Dwell-click detection                                               #
    # ------------------------------------------------------------------ #

    def update_dwell(self, fx: int, fy: int) -> str | None:
        """
        Call this every frame with the index finger position.
        Returns action string when dwell completes, else None.
        Actions: "color_0" … "color_8", "analyze", "undo", "clear", "save"
        """
        zone = self._hit_zone(fx, fy)

        if zone is None:
            self._dwell_target = None
            self._dwell_count  = 0
            return None

        if zone != self._dwell_target:
            self._dwell_target = zone
            self._dwell_count  = 0
            return None

        self._dwell_count += 1
        if self._dwell_count >= DWELL_FRAMES:
            self._dwell_count  = 0
            self._dwell_target = None
            return zone

        return None

    def draw_dwell_progress(self, frame: np.ndarray, fx: int, fy: int):
        """Overlay dwell progress arc over hovered zone."""
        if self._dwell_target is None or self._dwell_count == 0:
            return
        ratio   = self._dwell_count / DWELL_FRAMES
        angle   = int(360 * ratio)
        center  = self._zone_center(self._dwell_target)
        if center is None:
            return
        cx, cy = center
        axes   = (28, 28)
        cv2.ellipse(frame, (cx, cy), axes, -90, 0, angle, (0, 230, 120), 3)

    # ------------------------------------------------------------------ #
    #  Result overlay                                                      #
    # ------------------------------------------------------------------ #

    def draw_result_overlay(self, frame: np.ndarray, result, state_name: str):
        """Semi-transparent panel showing analysis output."""
        from korean_analyzer import AnalysisState

        h, w = frame.shape[:2]
        panel_x1, panel_y1 = 40, h // 5
        panel_x2, panel_y2 = w - 40, h - 60

        # Darken background
        overlay = frame.copy()
        cv2.rectangle(overlay, (panel_x1, panel_y1), (panel_x2, panel_y2), (10, 10, 20), -1)
        cv2.addWeighted(overlay, 0.82, frame, 0.18, 0, frame)

        # Border
        cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (60, 200, 120), 2)

        if state_name == "RUNNING":
            dots = "." * (1 + (self._frame_count // 10) % 3)
            pulse = abs(math.sin(self._frame_count * 0.1))
            g = int(100 + 120 * pulse)
            cv2.putText(frame, f"Analyzing{dots}", (panel_x1 + 40, panel_y1 + 80),
                        cv2.FONT_HERSHEY_DUPLEX, 1.1, (0, g, 80), 2)
            cv2.putText(frame, "Claude is reading your Korean strokes...",
                        (panel_x1 + 40, panel_y1 + 130),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (130, 130, 160), 1)
            return

        if state_name in ("DONE", "ERROR"):
            # Title
            cv2.putText(frame, "Korean Analysis", (panel_x1 + 30, panel_y1 + 50),
                        cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 230, 120), 2)

            fields = [
                ("Korean",        result.korean,        (200, 200, 255)),
                ("Romanization",  result.romanization,  (180, 255, 200)),
                ("Translation",   result.translation,   (255, 255, 180)),
                ("Notes",         result.notes,         (160, 160, 180)),
            ]

            y = panel_y1 + 110
            for label, value, col in fields:
                if not value:
                    continue
                cv2.putText(frame, f"{label}:", (panel_x1 + 30, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (120, 120, 140), 1)
                # Wrap long text
                for line in self._wrap(value, 55):
                    y += 30
                    cv2.putText(frame, line, (panel_x1 + 50, y), cv2.FONT_HERSHEY_SIMPLEX,
                                0.65, col, 1, cv2.LINE_AA)
                y += 18

            # Controls
            cv2.putText(frame,
                        "Press R to close  |  S to save canvas",
                        (panel_x1 + 30, panel_y2 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 100, 130), 1)

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _draw_swatches(self, frame: np.ndarray):
        for i, zone in enumerate(self._swatch_zones):
            x1, y1, x2, y2 = zone
            name, bgr = PALETTE[i]

            # Swatch fill
            cv2.rectangle(frame, (x1, y1), (x2, y2), bgr, -1)

            # Eraser gets a special border
            if name == "Eraser":
                cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 80, 80), 1)
                cv2.putText(frame, "X", (x1 + 17, y2 - 13),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (150, 150, 150), 2)

            # Selection highlight
            if i == self.selected_palette_idx:
                cv2.rectangle(frame, (x1 - 3, y1 - 3), (x2 + 3, y2 + 3), (0, 230, 120), 3)

    def _draw_buttons(self, frame: np.ndarray):
        buttons = [
            (self._analyze_zone, "ANALYZE", (0, 160, 60),   (0, 220, 100)),
            (self._undo_zone,    "UNDO",    (120, 80,  0),   (180, 120, 0)),
            (self._clear_zone,   "CLEAR",   (150, 0,   0),   (220, 40,  40)),
            (self._save_zone,    "SAVE",    (0,   80,  160),  (0,  140, 220)),
        ]
        for (x1, y1, x2, y2), label, dark, bright in buttons:
            cv2.rectangle(frame, (x1, y1), (x2, y2), dark, -1)
            tw, th = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)[0]
            tx = x1 + ((x2 - x1) - tw) // 2
            ty = y1 + ((y2 - y1) + th) // 2
            cv2.putText(frame, label, (tx, ty),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, bright, 1, cv2.LINE_AA)

    def _compute_swatch_zones(self) -> list:
        zones = []
        margin = 12
        y1 = (TOOLBAR_H - SWATCH_SZ) // 2
        y2 = y1 + SWATCH_SZ
        for i in range(len(PALETTE)):
            x1 = margin + i * (SWATCH_SZ + SWATCH_PAD)
            zones.append((x1, y1, x1 + SWATCH_SZ, y2))
        return zones

    def _compute_btn_zone(self, idx: int) -> tuple:
        bw, bh  = 90, 54
        margin  = 12
        right   = self.fw - margin
        y1      = (TOOLBAR_H - bh) // 2
        x2      = right - idx * (bw + 10)
        x1      = x2 - bw
        return (x1, y1, x2, y1 + bh)

    def _hit_zone(self, fx: int, fy: int) -> str | None:
        if fy > TOOLBAR_H:
            return None
        for i, (x1, y1, x2, y2) in enumerate(self._swatch_zones):
            if x1 <= fx <= x2 and y1 <= fy <= y2:
                return f"color_{i}"
        btn_map = [
            (self._analyze_zone, "analyze"),
            (self._undo_zone,    "undo"),
            (self._clear_zone,   "clear"),
            (self._save_zone,    "save"),
        ]
        for (x1, y1, x2, y2), name in btn_map:
            if x1 <= fx <= x2 and y1 <= fy <= y2:
                return name
        return None

    def _zone_center(self, zone_id: str) -> tuple | None:
        if zone_id.startswith("color_"):
            idx = int(zone_id.split("_")[1])
            x1, y1, x2, y2 = self._swatch_zones[idx]
            return ((x1 + x2) // 2, (y1 + y2) // 2)
        btn_map = {
            "analyze": self._analyze_zone,
            "undo":    self._undo_zone,
            "clear":   self._clear_zone,
            "save":    self._save_zone,
        }
        if zone_id in btn_map:
            x1, y1, x2, y2 = btn_map[zone_id]
            return ((x1 + x2) // 2, (y1 + y2) // 2)
        return None

    @staticmethod
    def _wrap(text: str, max_chars: int) -> list[str]:
        words = text.split()
        lines, line = [], ""
        for w in words:
            if len(line) + len(w) + 1 <= max_chars:
                line += ("" if not line else " ") + w
            else:
                if line:
                    lines.append(line)
                line = w
        if line:
            lines.append(line)
        return lines